In [48]:
import pandas as pd 
import copy
import Escape_score_predictor as esp
import matplotlib.pyplot as plt
import numpy as np
from Bio import SeqIO
import pandas as pd 

import tensorflow as tf
import tensorflow.keras.models as model
import Escape_score_predictor
import glob 

escape_threshold = 0.95

omicron_seq_path = '/home/perm/sars_escape_netv2/data/omicron/raw/omicron.fasta'
mut_save_path = "/home/perm/sars_escape_netv2/data/omicron/omicron_mutants.csv"
mut_window_save_path = '/home/perm/sars_escape_netv2/data/omicron/omicron_mutants_with_window.csv'
disc_model_path = '/home/perm/sars_escape_netv2/model/M3/sarsx_disc'
unfiltered_mut_path = '/home/perm/sars_escape_netv2/data/omicron/omicron_mutants_with_window_with_scores.csv' 
prob_esc_mut_path = '/home/perm/sars_escape_netv2/data/omicron/omicron_prob_escape_mutants.csv' 
filtered_prob_esc_mut_path = '/home/perm/sars_escape_netv2/data/omicron/omicron_filtered_prob_escape_mutants.csv' 


residues = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 
    'F', 'P', 'S', 'T', 'W', 'Y', 'V']

In [49]:
def read_omicron():
    with open(omicron_seq_path, 'r') as handle:
        for record in SeqIO.parse(handle, 'fasta'):
            id = record.id
            description = record.description
            seq = str(record.seq)
            # print(f'Id: {id} Desc: {description}')
            # print('Seq Length:'len(seq))
        return seq

def get_omicron_mutants():
    '''
    Give Omicron Seq of Length  1285 : MFVFLVLL...
    Step 1: Loop through Each Residue:
    1.1 : Mutate Each residue at that specific postion to all possible other Residues
    1.2 : List the Mutant information such that : M1C, M1D, M1Q and So on. 
    1.3 Save those mutant list to CSV 

    Step 2: Read CSV and Compute the Escape score
    2.1 Read CSV and Dynamically construct mutated Window Segement
    2.2 Tabulate the Escape Score 
    2.3 Save the mutant Score | Mutant | Mutant window


    '''
    mutatant_list = []
    spike_pos = 1
    seq =  read_omicron()
    for omicron_res in seq:
        for res in residues:
            #Avoid the mutant to be added into the mutant list if the residue is same as omicron_res
            #For each postion there will be 19 other mutants, if total residues considered ==20
            if res == omicron_res:
                continue 
            mutant = omicron_res+str(spike_pos)+res
            # print("The new Mutant is : ", mutant)
            mutatant_list.append(mutant)
        
        
        spike_pos += 1 

    #total number of mutants 
    total_mutants = len(seq) * 19
    print('Prospective total mutatns: ', total_mutants)
    assert total_mutants == len(mutatant_list)
    return mutatant_list

def save_omicron_muts(mutants, save_path):
    df = pd.DataFrame({'mutant': mutants })
    df.to_csv(save_path, index=False)
    print('Mutants Saved Successfully to ', save_path)
    
def construct_retrieve_window_from_mutant(mut_save_path):
    '''
    Reads mutant info Such as : E512K
    Reads Omicron Spike protein
    Mutates spike protein at specific postion P
    Extract mutated segement of lenght 20
    Returns windows segements along with mutant information : List (tuple(window, mutant))
    '''
    seq = read_omicron()
    df = pd.read_csv(mut_save_path)
    mutants = df['mutant'].to_list()
    window_len = 20
    windows = []
    for mutant in mutants:
        original_residue = mutant[0]
        changed_residue = mutant[-1]
        position = int(mutant[1:-1])
        # print('Mutant:', mutant)
        # print(f'Original {original_residue} Pos: {position} Changed: {changed_residue}')
        
        position = position - 1 #This is done becuase seqeunce array is 0 based
        #Extract the mutated window
        mutable_seq = None
        mutable_seq = [residue for residue in seq]
        mutable_seq[position] = changed_residue
        if position < window_len:
            window = mutable_seq[0: window_len]
        elif position > len(seq) - window_len:
            window = mutable_seq[len(seq) - 20 : len(seq)]
        else:
            window = mutable_seq[position-10: position+10]
        
        window_str= ''.join(window)
        # print(type(window_str))
        assert len(window_str) == window_len
        windows.append((window_str, mutant))
    return windows

def save_omicron_mutants_with_window(windows, csv_save_path):
    
    '''
    @param: windows: List of Tuple of window and Mutant) 
    Saves Omicron mutated windows along with its corresponding mutation information. 
    '''
    df1 = pd.DataFrame(windows, columns=['window', 'mutant'])
    df1.to_csv(csv_save_path, index=False)
    print('Window info saved successfully to ', csv_save_path)
        

                    
def execute_save_scores():
   
    df2 = pd.read_csv(mut_window_save_path)
    windows = df2['window'].to_list()
    # for window in windows:
    #     feature = Escape_score_predictor.get_single_feature(window)
    #     learned_features.append(feature)
    
    learned_features = Escape_score_predictor.get_features_for_seqs(windows)
      
    disc_model = model.load_model(disc_model_path)
    scores = disc_model.predict(learned_features)

    df2['scores'] = scores
    df2.to_csv(unfiltered_mut_path, index=False)
    print('Scores saved successfully to path: ', unfiltered_mut_path)
    return df2

def filter_save_omnicron_esc_mutants(unfiltered_mut_path, filtered_mut_path):
    df = pd.read_csv(unfiltered_mut_path)
    df_filtered =  df[ df['scores']>0.5]
    #Sorting dataframe : sort_values(by=['col1'], ascending=False) 
    df_filtered = df_filtered.sort_values(by=['scores'], ascending=False) 
    df_filtered.to_csv(filtered_mut_path, index=False)
    print('Filtered Escape Mutants are saved successfully to : ',filtered_mut_path)
    

def execute_main(mut_save_path, mut_window_save_path):
    '''mutants = get_omicron_mutants()
    save_omicron_muts(mutants, mut_save_path)
    windows = construct_retrieve_window_from_mutant(mut_save_path)
    save_omicron_mutants_with_window(windows, mut_window_save_path)
    df2 = execute_save_scores()'''
    filter_save_omnicron_esc_mutants(unfiltered_mut_path, prob_esc_mut_path)
    

def save_prob_escape_mut_after_removing_known_mut():
    file_list = glob.glob('/home/perm/sars_escape_netv2/data/known_mutants/*.csv')
    dfs = [pd.read_csv(file) for file in file_list]
    combined_df = pd.concat(dfs, ignore_index=True)
    print('Total Mutants After Combining all known escape Mutatns: ', combined_df.count())
    known_muts = combined_df['mutant'].to_list()
    known_muts = set(known_muts) #Conveting list to set 

    df2 = pd.read_csv(prob_esc_mut_path)
    prob_esc_muts = df2['mutant'].to_list()
    prob_esc_muts = set(prob_esc_muts)
    print(f'Probable Escape Mutants Len: ',len(prob_esc_muts))
    common_intersected_muts = prob_esc_muts.intersection(known_muts)
    #Exclude these insteresected element 
    filter_df2 =df2[~df2['mutant'].isin(common_intersected_muts)]
    print('After removing common known mutants:', filter_df2.count())
    filter_df2.to_csv(filtered_prob_esc_mut_path, index=False)
    print('Probable Escape Mutants After removing common known escape mutants were saved to:',filtered_prob_esc_mut_path)



In [50]:
save_prob_escape_mut_after_removing_known_mut()

Total Mutants After Combining all known escape Mutatns:  mutant    2405
dtype: int64
Probable Escape Mutants Len:  2308
After removing common known mutants: window    2302
mutant    2302
scores    2302
dtype: int64
Probable Escape Mutants After removing common known escape mutants were saved to: /home/perm/sars_escape_netv2/data/omicron/omicron_filtered_prob_escape_mutants.csv


window    2302
mutant    2302
scores    2302
dtype: int64